# 01 — Repository Setup

**Course:** 42913 Social and Information Network Analysis — UTS  
**Topic:** GraphRAG and Context Retrieval using Graph-Based Ranking and Link Prediction

---

## Project Overview

This project builds a **GraphRAG (Graph Retrieval-Augmented Generation) system** that frames context retrieval as a **link prediction problem** on a network graph. We implement and compare six graph scoring algorithms across two datasets.

| Family | Methods |
|--------|---------|
| **Heuristic** | Common Neighbours, Jaccard Coefficient, Adamic-Adar Index, Katz Index |
| **Probabilistic** | Global PageRank, Personalized PageRank |

| Dataset | Size | Task |
|---------|------|------|
| Wikipedia Vote Network | 7,115 nodes · 103,689 edges | Link prediction |
| HotpotQA (distractor) | 7,405 dev questions | Multi-hop retrieval |

**Core idea:** A predicted link (A → B) means "B is the most relevant context for A" — exactly the retrieval problem in GraphRAG systems.

---

## Notebook Workflow

| # | Notebook | What it does |
|---|----------|--------------|
| 01 | Repository Setup | Install libraries, verify environment |
| 02 | Dataset Exploration | Understand graph structure and data properties |
| 03 | Wiki-Vote Experiments | Run all 6 methods, evaluate ROC-AUC and Precision@K |
| 04 | HotpotQA Experiments | Graph-based multi-hop sentence retrieval |
| 05 | Universal Module | Reusable `retrieve()` function demo |


## Project File Structure

```
GraphRAG/
├── README.md                    ← Project overview and instructions
├── requirements.txt             ← Python dependencies
├── configs/config.yaml          ← Centralised hyperparameters
├── data/raw/                    ← Downloaded datasets
├── figures/                     ← All saved plots (PNG)
├── results/                     ← Evaluation tables (CSV)
├── notebooks/                   ← Five experiment notebooks
├── scripts/                     ← CLI runners for headless execution
└── src/
    ├── data_loader.py           ← Dataset download and loading
    ├── graph_builder.py         ← Graph construction utilities
    ├── heuristic_methods.py     ← CN, Jaccard, Adamic-Adar, Katz
    ├── probabilistic_methods.py ← PageRank, Personalized PageRank
    ├── evaluation.py            ← ROC-AUC, Precision@K, Average Precision
    ├── graphrag_retriever.py    ← Universal GraphRAG retrieval module
    └── utils.py                 ← Seeds, logging, I/O helpers
```

### Method-to-Dataset Mapping

| Method | Wiki-Vote (Link Prediction) | HotpotQA (Retrieval) |
|--------|:---------------------------:|:--------------------:|
| Common Neighbours | ✓ | ✓ |
| Jaccard Coefficient | ✓ | ✓ |
| Adamic-Adar | ✓ | ✓ |
| Katz Index | ✓ | — (graph too large for Katz) |
| PageRank | ✓ | ✓ |
| Personalized PageRank | ✓ | ✓ (best for retrieval) |


---
## Step 1 — Install Required Libraries

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "../requirements.txt", "-q"])
print("Installation complete.")

---
## Step 2 — Import and Verify Library Versions

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import scipy
import sklearn

print(f"Python       {sys.version.split()[0]}")
print(f"NumPy        {np.__version__}")
print(f"Pandas       {pd.__version__}")
print(f"NetworkX     {nx.__version__}")
print(f"Matplotlib   {matplotlib.__version__}")
print(f"SciPy        {scipy.__version__}")
print(f"Scikit-learn {sklearn.__version__}")

**Expected output (actual results from this environment):**
```
Python       3.11.4
NumPy        1.26.4
Pandas       3.0.2
NetworkX     3.6.1
Matplotlib   3.10.9
SciPy        1.17.1
Scikit-learn 1.8.0
```

**Library roles in this project:**
| Library | Used for |
|---------|----------|
| NetworkX | Graph construction, PageRank, subgraph operations |
| SciPy | Sparse matrix multiplication for Katz Index |
| scikit-learn | ROC-AUC, Average Precision scoring |
| NumPy / Pandas | Numerical operations and results tables |
| Matplotlib / Seaborn | All visualisations and saved figures |

---
## Step 3 — Set Random Seeds

A single global seed (42) controls all randomness in the project:
- The 80/20 train/test edge split in notebook 03
- The PPR source-node sampling in notebook 03
- Negative edge sampling for the test set

This guarantees **fully reproducible results** across every run.

In [ ]:
from src.utils import set_random_seeds, setup_logging

SEED = 42
set_random_seeds(SEED)
logger = setup_logging()
print(f"Random seed set to {SEED}")

---
## Step 4 — Create Project Directories

In [ ]:
from src.utils import create_project_dirs

create_project_dirs(base_dir=PROJECT_ROOT)
print("\nDirectory layout:")
for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in sorted(dirs)
               if not d.startswith(".") and d not in ("__pycache__", "venv", ".git")]
    level  = root.replace(PROJECT_ROOT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

**Expected output:** Directory tree showing all 9 project folders.
```
GraphRAG/
  configs/
  data/
    outputs/
    processed/
    raw/
  figures/
  notebooks/
  results/
  scripts/
  src/
```
After running all notebooks, `figures/` will contain 10+ PNG plots and `results/` will contain 2 CSV tables.

---
## Step 5 — Verify All Source Modules Import Correctly

In [ ]:
import importlib

modules_to_test = [
    "src.data_loader",
    "src.graph_builder",
    "src.heuristic_methods",
    "src.probabilistic_methods",
    "src.evaluation",
    "src.graphrag_retriever",
    "src.utils",
]

all_ok = True
for mod_name in modules_to_test:
    try:
        importlib.import_module(mod_name)
        print(f"  OK    {mod_name}")
    except Exception as e:
        print(f"  FAIL  {mod_name}: {e}")
        all_ok = False

print()
print("All modules imported successfully." if all_ok else "Some imports failed — see errors above.")

**Expected output:**
```
  OK    src.data_loader
  OK    src.graph_builder
  OK    src.heuristic_methods
  OK    src.probabilistic_methods
  OK    src.evaluation
  OK    src.graphrag_retriever
  OK    src.utils

All modules imported successfully.
```

All 7 modules successfully imported confirms:
- No missing dependencies
- No syntax errors in any source file
- The project root is correctly on the Python path

If any module shows `FAIL`, check that `pip install -r requirements.txt` completed without errors in Step 1.

---
## Setup Summary

| Step | What was done | Result |
|------|--------------|--------|
| 1 — Install | `pip install -r requirements.txt` | All packages installed |
| 2 — Versions | 7 core libraries verified | Python 3.11.4, NetworkX 3.6.1, scikit-learn 1.8.0 |
| 3 — Seeds | `set_random_seeds(42)` | Full reproducibility guaranteed |
| 4 — Directories | 9 project folders created | `figures/`, `results/`, `data/raw/` ready |
| 5 — Modules | 7 `src/` modules verified | All show `OK` |

The environment is ready. Proceed to **`02_dataset_exploration.ipynb`**.